# MiniMax H3 on Kaggle — One-Click Video Generation with Automatic Duration Extension

This notebook generates a video with MiniMax H3 + ComfyUI on a Kaggle free-tier GPU (T4×2 / T4×1 / P100, ~16 GB VRAM each), and can **automatically extend the result to any requested duration** — 10s, 15s, 60s, or longer — without exceeding the model's proven single-shot range and without risking CUDA out-of-memory errors.

**How extension works:** MiniMax H3's official ComfyUI node (`MiniMaxH3ImageToVideo`) is trained and reliable for roughly **5–15 seconds per single generation call** (124–362 frames at 24 fps; the node itself documents longer as "untested"). It does not expose a "continue in latent space" API — only an optional `first_frame` **image** conditioning input. So instead of one oversized, OOM-prone generation call, this notebook:

1. Splits the requested total duration into a sequence of **segments**, each sized at or below the model's proven-safe single-shot length (default ≤ 15s).
2. Generates segment 1 normally (text → video + audio).
3. Extracts the **last decoded frame** of each segment and feeds it into the next segment as `first_frame`, so the next segment continues visually from where the previous one ended.
4. Stitches every segment back together with FFmpeg into one final MP4, trimming the one duplicated hand-off frame (and the matching sliver of audio) at each seam so there is no stutter or A/V drift.

Because every single generation call always uses the exact same already-proven-safe resolution/duration/VRAM footprint — regardless of how long the **final** video ends up being — this scales to 60s+ without ever asking the GPU to do more work per step than the baseline ≤15s case already handles safely. A request at or below the safe single-shot ceiling runs as one ungapped generation (best quality, no seams at all); anything longer is chained automatically.

> A genuinely different, more reliable extension mechanism (e.g. true latent-space continuation) becomes available in a future ComfyUI H3 node? Swap it in inside `build_h3_workflow()` in the "Build the API workflow" section — the planning, retry, resumability, and stitching logic around it is written to be agnostic to *how* one segment is conditioned on the previous one; frame-conditioned chaining is used here because it is the mechanism this node actually exposes today, and it does not increase per-call VRAM/compute cost as the total duration grows.

MiniMax H3 generation is the only stage in this notebook — there is no separate upscaling pass. If you want to upscale the output afterward, do that in a separate notebook/tool of your choice.

## How to run the complete pipeline
1. In Kaggle, enable **Internet** and a **GPU** runtime (T4×2 recommended; a single T4/P100 also works).
2. Edit the prompt/duration/settings in the user configuration cell — or drive everything through `PARAMS` (see "Automation parameters" below).
3. Run all cells in order. Setup (install + model download + server start) happens once; the "Generate" cell then runs one or more chained segments automatically depending on the requested duration, and the "Stitch segments" cell assembles the final MP4.
4. Keep `/kaggle/working/minimax_h3_segments/` until the final video is verified. A rerun resumes from the last completed segment instead of starting over — this matters for long, high-duration jobs where a Kaggle session might take hours and could disconnect partway through.

> **Running this from external automation?** See "Automation parameters" right below — every value here (including `duration_seconds`) can be overridden via `params.json` without editing this notebook, and the single file an automation needs to read back is `/kaggle/working/automation_manifest.json`.

## Automation parameters (for API-driven / external-orchestrator runs)

Everything an external automation needs to control lives in **one** cell below (`DEFAULT_PARAMS`). Two ways to drive a run without touching any other cell:

- **A) Patch-before-push:** rewrite the dict between the `AUTOMATION_PARAMS_BEGIN` / `AUTOMATION_PARAMS_END` markers before calling `kaggle kernels push`.
- **B) Dataset override, no notebook edit:** attach/update a small Kaggle Dataset containing a `params.json` (same keys as `DEFAULT_PARAMS`) as a `dataset_sources` input. The loader below finds it automatically at `/kaggle/input/*/params.json` and merges it over the defaults — the notebook file itself never has to change between jobs.

Manual/interactive use is unaffected: just edit `DEFAULT_PARAMS` directly and run all cells as before.

**The one field that controls duration extension is `duration_seconds`.** Ask for `10`, `15`, `60` — whatever a caller (e.g. a Kaggle-CLI-driven frontend) needs right now — and the notebook automatically decides, from `max_single_shot_seconds`, how many chained segments are needed and generates + stitches them without any other change. Optional `chunk_seconds` and `segment_prompts` give finer control when you want it (see the advanced settings and generation cells for details).

Everything this run actually did (prompt, resolution, seed, segment plan, output file paths/sizes) is written at the very end to **`/kaggle/working/automation_manifest.json`** — that single file is all an external automation needs to read to know what to download.

In [1]:
# ============================================================
# AUTOMATION PARAMETERS
# ============================================================
#
# Target:
#   - Total duration: 30 seconds
#   - Segments: 3 x ~10 seconds
#   - Resolution: 352 x 608 (9:16 vertical)
#   - MiniMax H3 Turbo LoRA
#   - 4 sampling steps
#
# Flask embeds the per-job settings in this notebook before pushing it.
# For a manual Kaggle run, edit DEFAULT_PARAMS below.
# ============================================================

import json
from pathlib import Path


# ============================================================
# SEGMENT PROMPTS
# ============================================================

SEGMENT_1_PROMPT = """
integrated_multimodal_description:

[Segment 1, 0–10s]

Vertical 9:16 ultra-realistic live-action cinematic science-fiction thriller.

A woman in her late twenties stands alone beneath a covered bus stop on a quiet residential street at night. She has a realistic natural face, dark shoulder-length hair, realistic skin texture, realistic eyes, a dark wool coat, and a simple smartwatch on her left wrist.

CHARACTER CONTINUITY:
Keep exactly the same woman throughout the entire sequence. Preserve the same face, facial identity, hairstyle, hair length, dark wool coat, smartwatch, body proportions, skin texture, and overall silhouette. No character transformation, no clothing change, no hairstyle change, no age change, no duplicated person.

The street is quiet and mostly empty. Light mist drifts naturally through the environment. A single overhead streetlamp above her begins pulsing in an unusual rhythmic breathing pattern: bright, dim, bright, dim. It should feel like an electrical malfunction rather than a normal flicker.

The camera starts in a stable medium close-up at eye level, keeping her face clearly visible and detailed, then performs a very slow cinematic push-in.

She initially looks calm, then notices the strange lamp. She slowly raises her eyes toward it. Her expression changes naturally from neutral to slightly concerned.

Natural human motion, realistic facial expressions, anatomically correct hands and body, physically believable lighting, realistic nighttime exposure, subtle mist, shallow depth of field, subtle lens bloom, restrained film grain, realistic live-action cinematography.

No text, no subtitles, no logos, no additional characters, no visual distortion.

Audio:
Faint realistic nighttime ambience, distant traffic hum, light wind, occasional moisture dripping, subtle city ambience, and a low electrical buzz from the streetlamp. No music.

End with the woman looking upward at the pulsing lamp, ready for the next continuous segment.
""".strip()


SEGMENT_2_PROMPT = """
integrated_multimodal_description:

[Segment 2, 10–20s]

Continue seamlessly from the previous segment.

EXACT CONTINUITY:
Use exactly the same woman from Segment 1 with the same face, facial identity, hairstyle, hair length, dark wool coat, smartwatch, body proportions, skin texture, environment, bus stop, street, mist, nighttime lighting, and overall visual style.

Do not redesign or transform the character.

The woman slowly raises her left wrist into frame and looks closely at her smartwatch.

The smartwatch display shows a heart-rate reading around 78 BPM with a small pulsing heart icon.

The heart icon begins flashing with a subtle electronic pulse.

Approximately half a second later, the overhead streetlamp responds with the same pulse rhythm.

She looks from the smartwatch to the streetlamp, then back to the smartwatch.

Her facial expression naturally changes from curiosity to unease. Her breathing becomes slightly faster. Her eyes remain realistic and expressive.

She slowly takes one cautious step backward.

The camera performs a slow, controlled cinematic arc around her while keeping her face large enough in frame to preserve facial detail.

Maintain anatomically correct arms, wrists, hands, fingers, face, and body proportions. No extra limbs or distorted hands.

As the segment progresses, additional streetlamps farther down the residential street begin responding faintly to the same synchronization.

Realistic live-action cinematography, physically believable lighting, realistic skin texture, realistic eyes, natural hair movement, subtle handheld micro-movement, shallow depth of field, restrained film grain, subtle lens bloom.

No text, no subtitles, no logos, no additional people, no character transformation.

Audio:
Night ambience continues. The smartwatch emits a quiet electronic pulse whenever the heart icon flashes. Approximately half a second later, the streetlamp produces a subtle electrical click followed by its electrical hum. The woman's breathing becomes slightly faster. No music.

End with the woman staring toward the distant street as more lamps begin synchronizing.
""".strip()


SEGMENT_3_PROMPT = """
integrated_multimodal_description:

[Segment 3, 20–30s]

Continue seamlessly from the previous segment.

EXACT CONTINUITY:
Use exactly the same woman with the identical face, facial identity, hairstyle, dark wool coat, smartwatch, body proportions, environment, bus stop, mist, nighttime atmosphere, and lighting established in Segments 1 and 2.

Do not redesign, morph, age, duplicate, or transform the character.

The woman remains beneath the bus stop while the camera slowly begins to rise and pull backward.

One by one, the streetlamps stretching into the distance begin pulsing in exactly the same rhythm as the woman's smartwatch.

The synchronized pulses spread farther and farther down the street.

Soon dozens of streetlamps are pulsing together in perfect synchronization, creating a powerful wave of rhythmic illumination through the mist-covered street.

The woman looks upward in shock. Keep her face recognizable and visually stable.

The camera continues a smooth cinematic crane upward and backward, transitioning gradually from the medium shot into a wide cinematic establishing shot.

Reveal more of the residential street, mist, distant synchronized lamps, and realistic nighttime environment.

The electrical pulses become increasingly intense while remaining physically believable.

At the final synchronized pulse, every visible streetlamp suddenly turns completely dark at exactly the same moment.

Immediately cut to complete black.

The final visual moment is total darkness.

Audio:
The woman's breathing continues. Electrical clicking and buzzing gradually multiply as more streetlamps synchronize. The electrical atmosphere grows larger and more intense. At the final synchronized flash, every electrical sound stops instantly. Environmental sound also stops. Complete silence immediately before the cut to black.

No non-diegetic music.

Ultra-realistic live-action science-fiction thriller, high-budget cinematic photography, realistic human anatomy, stable facial identity, stable clothing, realistic mist, physically accurate lighting, realistic camera motion, subtle lens bloom, cinematic depth of field, restrained film grain.

No text, no subtitles, no logos, no additional characters, no visual distortion.
""".strip()


# ============================================================
# DEFAULT PARAMETERS
# ============================================================

DEFAULT_PARAMS = {

    # --------------------------------------------------------
    # JOB
    # --------------------------------------------------------
    "job_id": "h3-30s-night-lamp",

    # Fallback/default prompt.
    # Segment prompts below are used for the 3 individual chunks.
    "prompt": SEGMENT_1_PROMPT,

    # --------------------------------------------------------
    # RESOLUTION
    # --------------------------------------------------------
    # 352 x 608 = 9:16 vertical
    #
    # Width  = 352
    # Height = 608
    #
    # Both are divisible by 32.
    # --------------------------------------------------------
    "resolution_preset": "Custom",
    "custom_width": 352,
    "custom_height": 608,

    # --------------------------------------------------------
    # TURBO
    # --------------------------------------------------------
    "use_turbo_lora": True,
    "turbo_steps": 4,

    # General steps.
    # Keep this at 4 when using the 4-step Turbo workflow.
    "steps": 4,

    # --------------------------------------------------------
    # TOTAL DURATION
    # --------------------------------------------------------
    "duration_seconds": 30.0,

    # Fixed seed makes testing/reproducing runs easier.
    "seed": 20260803,

    # --------------------------------------------------------
    # SEGMENT PROMPTS
    # --------------------------------------------------------
    # IMPORTANT:
    # The 30 seconds are explicitly divided into:
    #
    #   Segment 1 = 0-10s
    #   Segment 2 = 10-20s
    #   Segment 3 = 20-30s
    #
    # Each segment has its own prompt so the story progresses
    # instead of restarting the same scene every time.
    # --------------------------------------------------------
    "segment_prompts": [
        SEGMENT_1_PROMPT,
        SEGMENT_2_PROMPT,
        SEGMENT_3_PROMPT,
    ],

    # --------------------------------------------------------
    # CHUNKING
    # --------------------------------------------------------
    #
    # Force 10-second chunks instead of letting the workflow
    # automatically divide the 30 seconds.
    #
    # Result:
    #   30s -> 10s + 10s + 10s
    # --------------------------------------------------------
    "max_single_shot_seconds": 10.0,
    "chunk_seconds": 10.0,

    # --------------------------------------------------------
    # SERVER RESTART
    # --------------------------------------------------------
    #
    # There are only 3 chunks, so there is no need to restart
    # ComfyUI during this job.
    #
    # 0 / None = never restart automatically.
    # --------------------------------------------------------
    "restart_server_every_n_chunks": 0,

    # --------------------------------------------------------
    # TEXT ENCODER GPU
    # --------------------------------------------------------
    #
    # One T4 setup.
    # --------------------------------------------------------
    "use_second_t4_for_text_encoder": False,

    # --------------------------------------------------------
    # FALLBACK / COMFYUI
    # --------------------------------------------------------
    "auto_fallback": True,

    # Keep your current ComfyUI reference.
    # For a production automation pipeline, consider pinning
    # this later to a known-good commit/version.
    "comfy_ref": "master",

    # Set this only when the workflow/model requires HF auth.
    "hf_token": None,

    # --------------------------------------------------------
    # FINAL VIDEO ENCODING
    # --------------------------------------------------------
    #
    # These settings matter when 2+ generated segments are
    # stitched together.
    # --------------------------------------------------------
    "video_crf": 18,
    "video_preset": "slow",
    "video_pixel_format": "yuv420p",
    "video_faststart": True,
}


# ============================================================
# BUILD PARAMETERS
# ============================================================

# Flask embeds this job's exact inputs into the pushed notebook immediately
# after the marker below. Manual Kaggle runs use DEFAULT_PARAMS.
PARAMS = dict(DEFAULT_PARAMS)
known_overrides = {}
PARAMS.update(known_overrides)

# Optional signed progress callback used by the Flask automation service.
# When no callback endpoint is configured, this notebook still runs normally.
import time as _automation_time
_AUTOMATION_PROGRESS_LAST_SENT = 0.0
_AUTOMATION_PROGRESS_LAST_KEY = None

def report_automation_progress(stage, message, segment_index, completed_segments, total_segments, segment_progress):
    callback_url = PARAMS.get("automation_progress_url")
    callback_token = PARAMS.get("automation_progress_token")
    if not callback_url or not callback_token:
        return

    total_segments = max(1, int(total_segments))
    completed_segments = min(total_segments, max(0, int(completed_segments)))
    segment_progress = min(1.0, max(0.0, float(segment_progress)))
    percent = min(99, round(100 * (completed_segments + segment_progress) / total_segments))
    key = (percent, str(stage))
    now = _automation_time.monotonic()
    global _AUTOMATION_PROGRESS_LAST_SENT, _AUTOMATION_PROGRESS_LAST_KEY
    if now - _AUTOMATION_PROGRESS_LAST_SENT < 0.75:
        return

    payload = {
        "stage": str(stage),
        "message": str(message),
        "segment_index": int(segment_index),
        "completed_segments": completed_segments,
        "total_segments": total_segments,
        "segment_progress": segment_progress,
    }
    _AUTOMATION_PROGRESS_LAST_SENT = now
    _AUTOMATION_PROGRESS_LAST_KEY = key
    try:
        import requests as _automation_requests
        _automation_requests.post(
            callback_url,
            json=payload,
            headers={"X-Kaggle-Progress-Token": callback_token},
            timeout=(1.5, 2.5),
        ).raise_for_status()
    except Exception:
        # Progress delivery must never interrupt model generation.
        pass



# ============================================================
# VALIDATE IMPORTANT SETTINGS
# ============================================================

if PARAMS["duration_seconds"] <= 0:
    raise ValueError(
        "duration_seconds must be greater than 0."
    )

if PARAMS["chunk_seconds"] is not None:
    if PARAMS["chunk_seconds"] <= 0:
        raise ValueError(
            "chunk_seconds must be greater than 0."
        )

if PARAMS["custom_width"] <= 0 or PARAMS["custom_height"] <= 0:
    raise ValueError(
        "custom_width and custom_height must be greater than 0."
    )

if PARAMS["steps"] <= 0:
    raise ValueError(
        "steps must be greater than 0."
    )

if PARAMS["use_turbo_lora"] and PARAMS["turbo_steps"] <= 0:
    raise ValueError(
        "turbo_steps must be greater than 0 when Turbo LoRA is enabled."
    )


# ============================================================
# INFORMATION OUTPUT
# ============================================================

print("Using DEFAULT_PARAMS and any job parameters embedded by Flask.")


# ============================================================
# FINAL CONFIG SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL AUTOMATION CONFIGURATION")
print("=" * 60)

print(f"job_id:                     {PARAMS.get('job_id')}")

print(
    f"resolution:                 "
    f"{PARAMS.get('custom_width')}x{PARAMS.get('custom_height')}"
)

print(
    f"duration_seconds:           "
    f"{PARAMS.get('duration_seconds')}"
)

print(
    f"chunk_seconds:              "
    f"{PARAMS.get('chunk_seconds')}"
)

print(
    f"max_single_shot_seconds:    "
    f"{PARAMS.get('max_single_shot_seconds')}"
)

print(
    f"expected segments:          "
    f"{int(PARAMS.get('duration_seconds') / PARAMS.get('chunk_seconds'))}"
)

print(
    f"use_turbo_lora:             "
    f"{PARAMS.get('use_turbo_lora')}"
)

print(
    f"turbo_steps:                "
    f"{PARAMS.get('turbo_steps')}"
)

print(
    f"steps:                      "
    f"{PARAMS.get('steps')}"
)

print(
    f"seed:                       "
    f"{PARAMS.get('seed')}"
)

print(
    f"use_second_t4_for_encoder: "
    f"{PARAMS.get('use_second_t4_for_text_encoder')}"
)

print(
    f"restart_every_n_chunks:     "
    f"{PARAMS.get('restart_server_every_n_chunks')}"
)

print(
    f"segment_prompts:            "
    f"{len(PARAMS.get('segment_prompts', []))}"
)

print(
    f"video_crf:                  "
    f"{PARAMS.get('video_crf')}"
)

print(
    f"video_preset:               "
    f"{PARAMS.get('video_preset')}"
)

print(
    f"video_pixel_format:         "
    f"{PARAMS.get('video_pixel_format')}"
)

print(
    f"video_faststart:            "
    f"{PARAMS.get('video_faststart')}"
)

print("=" * 60)

print("\n✓ Configuration ready.")
print("✓ Target: 30s total")
print("✓ Target segmentation: 3 x 10s")
print("✓ Target aspect ratio: 9:16")
print("✓ Turbo: 4 steps")

ℹ No external params.json found — using DEFAULT_PARAMS.

FINAL AUTOMATION CONFIGURATION
job_id:                     h3-30s-night-lamp
resolution:                 352x608
duration_seconds:           30.0
chunk_seconds:              10.0
max_single_shot_seconds:    10.0
expected segments:          3
use_turbo_lora:             True
turbo_steps:                4
steps:                      4
seed:                       20260803
use_second_t4_for_encoder: False
restart_every_n_chunks:     0
segment_prompts:            3
video_crf:                  18
video_preset:               slow
video_pixel_format:         yuv420p
video_faststart:            True

✓ Configuration ready.
✓ Target: 30s total
✓ Target segmentation: 3 x 10s
✓ Target aspect ratio: 9:16
✓ Turbo: 4 steps


<div style="
    border:2px solid #76d65d;
    border-left:10px solid #72ff57;
    border-radius:10px;
    padding:18px 22px;
    margin:18px 0 10px 0;
    background:linear-gradient(90deg, rgba(74,130,62,0.20), rgba(74,130,62,0.04));
">
<div style="font-size:24px;font-weight:800;letter-spacing:0.02em;">
1. USER CONFIGURATION — EDIT THE NEXT CELL
</div>
<div style="margin-top:7px;font-size:15px;">
Most users only need to change the prompt, resolution preset, duration, steps, and seed.
Set <code>duration_seconds</code> to anything — the notebook automatically decides whether that needs one generation or several chained ones. Advanced runtime and extension settings are kept in a separate cell below.
</div>
</div>

In [2]:
# ╔══════════════════════════════════════════════════════╗
# ║                 USER CONFIGURATION — EDIT HERE              ║
# ╚═════════════════════════════════════════════════════╝
#
# Manual/interactive use: edit DEFAULT_PARAMS in the "Automation
# parameters" cell above and re-run — nothing below needs to change.
#
# Automated use: everything below is driven by PARAMS (see the cell
# above). This cell itself never needs to be touched by automation.

PROMPT = str(PARAMS["prompt"]).strip()

# Optional per-segment prompt overrides for long, multi-segment
# extensions. None = reuse PROMPT for every segment.
SEGMENT_PROMPTS = PARAMS.get("segment_prompts")

# Pick one preset. "Custom" uses CUSTOM_WIDTH and CUSTOM_HEIGHT below.
RESOLUTION_PRESET = PARAMS["resolution_preset"]

RESOLUTION_PRESETS = {
    "Fast preview · 512×288": (512, 288),
    "Kaggle safe · 608×352": (608, 352),
    "Detailed preview · 736×416": (736, 416),
    "Custom": None,
}

CUSTOM_WIDTH = int(PARAMS["custom_width"])
CUSTOM_HEIGHT = int(PARAMS["custom_height"])

# --- Turbo Acceleration Config ---
USE_TURBO_LORA = bool(PARAMS["use_turbo_lora"])   # ~2.5-4x faster; keep True unless you need max fidelity
TURBO_STEPS = int(PARAMS["turbo_steps"])          # 6 = faster while retaining better quality than 4

# --- Total requested duration of the FINAL stitched video ---
# Any value is accepted. <= MAX_SINGLE_SHOT_SECONDS (advanced settings
# cell, default 15s) runs as a single generation. Longer values are
# split into chained segments automatically — see "Build the API
# workflow" and "Generate" further down.
DURATION_SECONDS = float(PARAMS["duration_seconds"])

STEPS = int(PARAMS["steps"])
SEED = int(PARAMS["seed"])

if USE_TURBO_LORA:
    print(f"⚡ Turbo LoRA enabled: overriding default STEPS={STEPS} with TURBO_STEPS={TURBO_STEPS}")
    STEPS = TURBO_STEPS

if RESOLUTION_PRESET not in RESOLUTION_PRESETS:
    raise ValueError(f"Unknown RESOLUTION_PRESET: {RESOLUTION_PRESET}")

if RESOLUTION_PRESET == "Custom":
    WIDTH, HEIGHT = int(CUSTOM_WIDTH), int(CUSTOM_HEIGHT)
else:
    WIDTH, HEIGHT = RESOLUTION_PRESETS[RESOLUTION_PRESET]

⚡ Turbo LoRA enabled: overriding default STEPS=4 with TURBO_STEPS=4


### Advanced runtime settings

These defaults are intended for Kaggle T4×2. Most users can leave this section unchanged. This is also where the automatic-duration-extension knobs live: `MAX_SINGLE_SHOT_SECONDS`, `CHUNK_SECONDS`, and `RESTART_SERVER_EVERY_N_CHUNKS`.

In [3]:
# ============================================================
# ADVANCED RUNTIME SETTINGS
# ============================================================

# Keep False initially.
# ComfyUI low-VRAM/offload is safer than forcing a ~15 GB
# Qwen3-VL encoder onto a 15 GB T4.
USE_SECOND_T4_FOR_TEXT_ENCODER = bool(PARAMS["use_second_t4_for_text_encoder"])

# Retry a failed segment at reduced settings only for genuine CUDA
# out-of-memory failures (not other errors).
AUTO_FALLBACK = bool(PARAMS["auto_fallback"])

JOB_ID = str(PARAMS.get("job_id") or "manual-run")

SAVE_PREFIX = f"MiniMax_H3/h3_kaggle_{JOB_ID}"

SERVER_PORT = 8188

# Pin this to a specific commit SHA once your automation is stable —
# "master" can drift upstream and silently change/break the workflow
# between automated runs.
COMFY_REF = str(PARAMS.get("comfy_ref") or "master")

# Public Comfy-Org model repo does not require an HF token.
HF_TOKEN = PARAMS.get("hf_token") or None

# ------------------------------------------------------------
# Duration extension settings
# ------------------------------------------------------------

# The model's proven, trained-safe length for a SINGLE generation call
# (MiniMaxH3ImageToVideo reports a trained range of ~124-362 frames,
# i.e. ~5.2s-15.1s at 24 fps; longer is officially "untested"). Any
# request at or below this runs as one segment. Anything above it is
# split into this many seconds (or fewer) per chained segment.
# Raising this beyond ~15s trades reliability/quality for fewer seams.
MAX_SINGLE_SHOT_SECONDS = float(PARAMS.get("max_single_shot_seconds") or 15.0)

# Force an exact per-segment length instead of the automatic even
# split (None = automatic). Rarely needed.
CHUNK_SECONDS = PARAMS.get("chunk_seconds")
CHUNK_SECONDS = float(CHUNK_SECONDS) if CHUNK_SECONDS else None

# Fully restart the ComfyUI process after every N completed segments
# (0/None = never). This resets GPU/RAM state between segments, which
# matters most on very long (many-segment, multi-hour) jobs where
# small fragmentation/leaks could otherwise accumulate toward an OOM.
RESTART_SERVER_EVERY_N_CHUNKS = int(PARAMS.get("restart_server_every_n_chunks") or 0)

# --- Output encoding (used only when 2+ segments are stitched into one
# final video; a single-segment run is copied through with zero
# re-encode, so these have no effect on short/default runs). ---
VIDEO_CRF = int(PARAMS["video_crf"])
VIDEO_PRESET = str(PARAMS["video_preset"])
VIDEO_PIXEL_FORMAT = str(PARAMS["video_pixel_format"])
VIDEO_FASTSTART = bool(PARAMS["video_faststart"])

In [4]:
import math

from IPython.display import HTML, display


def snap_h3_frames(seconds: float) -> int:
    """Round a requested duration (at 24 fps) up to the H3 model's frame
    grid: frame_count must satisfy frame_count % 17 == 5 (matching the
    MiniMaxH3ImageToVideo node's length input: min=5, step=17)."""
    requested = max(5, round(float(seconds) * 24))
    return requested + (5 - (requested % 17)) % 17


def frames_to_seconds(frames: int, fps: float = 24.0) -> float:
    return frames / fps


def plan_segments(total_seconds, max_single_shot_seconds, chunk_seconds, fps=24.0):
    """
    Decide how many chained segments are needed to reach `total_seconds`
    of final video, and how long each one should be.

    - If chunk_seconds is set explicitly, every segment uses that exact
      requested length (the last segment absorbs any remainder).
    - Otherwise:
        * total_seconds already fits in ONE segment within
          max_single_shot_seconds -> a single segment, no chaining.
          This is the best-quality path (no hand-off seams at all) and
          matches the model's own trained-safe single-shot range.
        * otherwise -> split into the smallest number of roughly-equal
          segments that each stay at/below max_single_shot_seconds, and
          chain them with first-frame continuation.
    """
    total_seconds = float(total_seconds)
    max_single_shot_seconds = float(max_single_shot_seconds)

    if chunk_seconds:
        chunk_seconds = float(chunk_seconds)
        lengths, remaining = [], total_seconds
        while remaining > 1e-6:
            lengths.append(min(chunk_seconds, remaining))
            remaining -= chunk_seconds
        lengths = lengths or [total_seconds]
    elif total_seconds <= max_single_shot_seconds + 1e-6:
        lengths = [total_seconds]
    else:
        num_segments = math.ceil(total_seconds / max_single_shot_seconds)
        lengths = [total_seconds / num_segments] * num_segments

    segments = []
    for i, requested in enumerate(lengths):
        frames = snap_h3_frames(requested)
        segments.append({
            "index": i,
            "requested_seconds": requested,
            "frames": frames,
            "actual_seconds": frames_to_seconds(frames, fps),
            "is_first": i == 0,
        })
    return segments


if WIDTH % 32 != 0 or HEIGHT % 32 != 0:
    raise ValueError("WIDTH and HEIGHT must both be divisible by 32.")

if STEPS < 1:
    raise ValueError("STEPS must be at least 1.")

SEGMENTS = plan_segments(DURATION_SECONDS, MAX_SINGLE_SHOT_SECONDS, CHUNK_SECONDS)
TOTAL_ACTUAL_SECONDS = sum(s["actual_seconds"] for s in SEGMENTS)

_chain_rows = "".join(
    f"<div style='margin-top:4px;'>"
    f"<b>Segment {s['index'] + 1}/{len(SEGMENTS)}</b>: {s['actual_seconds']:.2f}s "
    f"({s['frames']} frames)"
    f"{' · continues from previous segment' if not s['is_first'] else ''}"
    f"</div>"
    for s in SEGMENTS
)

display(HTML(f"""
<div style="
    border:1px solid #77b867;
    border-radius:9px;
    padding:14px 18px;
    margin:8px 0 18px 0;
    background:rgba(70,120,60,0.09);
    font-family:system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
">
  <div style="font-size:17px;font-weight:750;margin-bottom:10px;">
    Selected MiniMax H3 profile
  </div>

  <div style="display:grid;grid-template-columns:repeat(4,minmax(140px,1fr));gap:10px;">
    <div><span style="opacity:.68;">Canvas</span><br><b>{WIDTH} × {HEIGHT}</b></div>
    <div><span style="opacity:.68;">Requested duration</span><br><b>{DURATION_SECONDS:.2f}s</b></div>
    <div><span style="opacity:.68;">Planned duration</span><br><b>{TOTAL_ACTUAL_SECONDS:.2f}s</b> ({len(SEGMENTS)} segment{'s' if len(SEGMENTS) != 1 else ''})</div>
    <div><span style="opacity:.68;">Sampling</span><br><b>{STEPS} steps</b></div>
    <div><span style="opacity:.68;">Turbo</span><br><b>{'enabled' if USE_TURBO_LORA else 'disabled'}</b></div>
    <div><span style="opacity:.68;">GPU-1 text encoder</span><br><b>{'enabled' if USE_SECOND_T4_FOR_TEXT_ENCODER else 'disabled'}</b></div>
    <div><span style="opacity:.68;">Auto fallback</span><br><b>{'enabled' if AUTO_FALLBACK else 'disabled'}</b></div>
    <div><span style="opacity:.68;">Server recycle</span><br><b>{f'every {RESTART_SERVER_EVERY_N_CHUNKS} segments' if RESTART_SERVER_EVERY_N_CHUNKS else 'disabled'}</b></div>
  </div>

  <div style="margin-top:12px;">{_chain_rows}</div>
</div>
"""))

## 2. Hardware, memory, and disk preflight

In [5]:
from pathlib import Path
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time

try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is unavailable in this Kaggle image.") from exc

IS_KAGGLE = Path("/kaggle").exists()

SESSION_ROOT = (
    Path("/kaggle/temp/minimax_h3_session")
    if IS_KAGGLE
    else Path("/tmp/minimax_h3_session")
)

COMFY_DIR = SESSION_ROOT / "ComfyUI"

COMFY_OUTPUT_DIR = (
    Path("/kaggle/working/minimax_h3_output")
    if IS_KAGGLE
    else SESSION_ROOT / "output"
)

COMFY_TEMP_DIR = SESSION_ROOT / "temp"

FINAL_OUTPUT_DIR = (
    Path("/kaggle/working")
    if IS_KAGGLE
    else SESSION_ROOT / "final"
)

LOG_PATH = FINAL_OUTPUT_DIR / "minimax_h3_comfyui.log"

for directory in (
    SESSION_ROOT,
    COMFY_OUTPUT_DIR,
    COMFY_TEMP_DIR,
    FINAL_OUTPUT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


def gib(num_bytes: int) -> float:
    return num_bytes / (1024 ** 3)


print("=" * 70)
print("HARDWARE / ENVIRONMENT")
print("=" * 70)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible CUDA devices:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required. Enable the Kaggle T4 × 2 accelerator."
    )

for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(
        f"GPU {index}: {props.name} — "
        f"{gib(props.total_memory):.2f} GiB"
    )

if (
    torch.cuda.device_count() < 2
    and USE_SECOND_T4_FOR_TEXT_ENCODER
):
    print(
        "Only one GPU is visible; disabling GPU-1 text encoder."
    )
    USE_SECOND_T4_FOR_TEXT_ENCODER = False

disk_anchor = Path("/kaggle/temp" if IS_KAGGLE else "/tmp")
usage = shutil.disk_usage(disk_anchor)

print(
    f"Free disk at {disk_anchor}: "
    f"{gib(usage.free):.2f} GiB"
)

# Official H3 payload is large. Keep enough room for models,
# ComfyUI files, temporary data, and per-segment output videos.
# (No frame-by-frame upscaling storage is needed in this pipeline.)
MIN_RECOMMENDED_FREE_GIB = 40.0

if gib(usage.free) < MIN_RECOMMENDED_FREE_GIB:
    raise RuntimeError(
        f"Only {gib(usage.free):.2f} GiB free. "
        f"At least {MIN_RECOMMENDED_FREE_GIB:.1f} GiB "
        f"is recommended."
    )

if WIDTH % 32 != 0 or HEIGHT % 32 != 0:
    raise ValueError(
        "WIDTH and HEIGHT must both be divisible by 32."
    )

print("\nNVIDIA status:")
subprocess.run(["nvidia-smi"], check=False)

HARDWARE / ENVIRONMENT
Python: 3.12.13
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
PyTorch: 2.10.0+cu128
CUDA available: True
Visible CUDA devices: 2
GPU 0: Tesla T4 — 14.56 GiB
GPU 1: Tesla T4 — 14.56 GiB
Free disk at /kaggle/temp: 1024.67 GiB

NVIDIA status:
Thu Sep 24 12:56:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04

CompletedProcess(args=['nvidia-smi'], returncode=0)

## 3. Install ComfyUI and Python dependencies

In [6]:
import subprocess
import sys
from pathlib import Path


def run_checked(command, cwd=None, env=None):
    command = [str(x) for x in command]

    print("+", " ".join(command))

    subprocess.run(
        command,
        cwd=cwd,
        env=env,
        check=True,
    )


# Always start from a valid working directory.
Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working")


# ------------------------------------------------------------
# Clone/update ComfyUI
# ------------------------------------------------------------
if not (COMFY_DIR / ".git").exists():
    run_checked([
        "git",
        "clone",
        "--filter=blob:none",
        "https://github.com/Comfy-Org/ComfyUI.git",
        str(COMFY_DIR),
    ])

run_checked(
    [
        "git",
        "fetch",
        "--depth",
        "1",
        "origin",
        COMFY_REF,
    ],
    cwd=COMFY_DIR,
)

run_checked(
    [
        "git",
        "checkout",
        "--force",
        "FETCH_HEAD",
    ],
    cwd=COMFY_DIR,
)


# ------------------------------------------------------------
# Install ComfyUI dependencies
# ------------------------------------------------------------
run_checked([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(COMFY_DIR / "requirements.txt"),
])


# Notebook helpers
run_checked([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "huggingface_hub>=1.3.0,<2.0",
    "requests>=2.31.0",
    "psutil>=5.9.0",
    "websocket-client>=1.8.0",
    "tqdm>=4.66.0",
])

# ------------------------------------------------------------
# Turbo custom node
# ------------------------------------------------------------
turbo_node_dir = (
    COMFY_DIR
    / "custom_nodes"
    / "ComfyUI-MiniMax-H3-Turbo"
)

if not turbo_node_dir.exists():
    run_checked([
        "git",
        "clone",
        "--filter=blob:none",
        "https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git",
        str(turbo_node_dir),
    ])
else:
    print("Turbo node already exists.")

# ------------------------------------------------------------
# Clean pip cache
# ------------------------------------------------------------
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "cache",
        "purge",
    ],
    check=False,
)

COMFY_COMMIT = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=COMFY_DIR,
    text=True,
).strip()

print("\nComfyUI commit:", COMFY_COMMIT)

+ git clone --filter=blob:none https://github.com/Comfy-Org/ComfyUI.git /kaggle/temp/minimax_h3_session/ComfyUI


Cloning into '/kaggle/temp/minimax_h3_session/ComfyUI'...
Updating files: 100% (1222/1222), done.


+ git fetch --depth 1 origin master


From https://github.com/Comfy-Org/ComfyUI
 * branch              master     -> FETCH_HEAD


+ git checkout --force FETCH_HEAD


Note: switching to 'FETCH_HEAD'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 1568e6cf Lower memory usage and .comfy_attention support for lumina family models. (#16515)


+ /usr/bin/python3 -m pip install -q -r /kaggle/temp/minimax_h3_session/ComfyUI/requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.2/25.2 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Cloning into '/kaggle/temp/minimax_h3_session/ComfyUI/custom_nodes/ComfyUI-MiniMax-H3-Turbo'...


+ git clone --filter=blob:none https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git /kaggle/temp/minimax_h3_session/ComfyUI/custom_nodes/ComfyUI-MiniMax-H3-Turbo
Files removed: 156

ComfyUI commit: 1568e6cfd04586a4b3c4e1817ea7dde09b1bf9e7


## 4. Add an experimental GPU-1 text-encoder loader

Current native ComfyUI exposes `default` and `cpu` in the standard CLIP loader.
This tiny local node adds an explicit CUDA device option so the 15.x GB Qwen3-VL text encoder can optionally be pinned to a second T4 instead of sharing VRAM with the diffusion model.

In [7]:
from pathlib import Path

custom_node_dir = (
    COMFY_DIR
    / "custom_nodes"
    / "h3_kaggle_multigpu"
)

custom_node_dir.mkdir(
    parents=True,
    exist_ok=True,
)

custom_node_code = r'''
import torch
import folder_paths
import comfy.sd


class H3KaggleCLIPLoader:

    @classmethod
    def INPUT_TYPES(cls):
        devices = ["default", "cpu"]

        if torch.cuda.device_count() > 1:
            devices.append("cuda:1")

        return {
            "required": {
                "clip_name": (
                    folder_paths.get_filename_list("text_encoders"),
                ),
                "device": (devices,),
            }
        }

    RETURN_TYPES = ("CLIP",)
    FUNCTION = "load_clip"
    CATEGORY = "loaders/H3 Kaggle"

    def load_clip(self, clip_name, device):

        if device == "default":
            device = None

        elif device.startswith("cuda:"):
            index = int(device.split(":", 1)[1])

            if index >= torch.cuda.device_count():
                raise RuntimeError(
                    f"Requested {device}, but only "
                    f"{torch.cuda.device_count()} CUDA device(s) "
                    f"are visible."
                )

        if device is None:
            model_options = {}
        else:
            model_options = {
                "load_device": torch.device(device),
                "offload_device": torch.device("cpu"),
            }

        clip_path = folder_paths.get_full_path_or_raise(
            "text_encoders",
            clip_name,
        )

        clip_type = getattr(
            comfy.sd.CLIPType,
            "MINIMAX",
        )

        clip = comfy.sd.load_clip(
            ckpt_paths=[clip_path],
            embedding_directory=folder_paths.get_folder_paths(
                "embeddings"
            ),
            clip_type=clip_type,
            model_options=model_options,
        )

        return (clip,)


NODE_CLASS_MAPPINGS = {
    "H3KaggleCLIPLoader": H3KaggleCLIPLoader,
}

NODE_DISPLAY_NAME_MAPPINGS = {
    "H3KaggleCLIPLoader":
        "H3 Kaggle CLIP Loader (Experimental Multi-GPU)",
}
'''

node_file = custom_node_dir / "__init__.py"

node_file.write_text(
    custom_node_code,
    encoding="utf-8",
)

print("Wrote:", node_file)

Wrote: /kaggle/temp/minimax_h3_session/ComfyUI/custom_nodes/h3_kaggle_multigpu/__init__.py


## 5. Download the four official ComfyUI H3 model files

In [8]:
# ============================================================
# MINI­MAX H3 OFFICIAL MODEL DOWNLOADS
# ============================================================

import os
import json
from pathlib import Path

from huggingface_hub import hf_hub_download


# ------------------------------------------------------------
# Define ALL paths/functions BEFORE using them
# ------------------------------------------------------------

models_root = COMFY_DIR / "models"
models_root.mkdir(
    parents=True,
    exist_ok=True,
)


def file_gib(path: Path) -> float:
    return path.stat().st_size / (1024 ** 3)


os.environ.setdefault(
    "HF_HOME",
    str(SESSION_ROOT / "hf_home"),
)

os.environ.setdefault(
    "HF_HUB_ENABLE_HF_TRANSFER",
    "0",
)


# ------------------------------------------------------------
# Official repositories
# ------------------------------------------------------------

MODEL_REPO = "Comfy-Org/MiniMax-H3"

TURBO_REPO = "larryvrh/MiniMax-H3-Turbo-Lora"

TURBO_LORA_FILE = (
    "minimax_h3_turbo_v4_step600_ema.safetensors"
)


# ------------------------------------------------------------
# Official H3 files for T2VA / FL2VA
# ------------------------------------------------------------

MODEL_FILES = {
    "diffusion_models/"
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": 18.0,

    "text_encoders/"
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 14.0,

    "vae/"
    "minimax_h3_video_vae_fp16.safetensors": 4.5,

    "vae/"
    "minimax_h3_audio_vae_fp32.safetensors": 0.50,
}


# ============================================================
# 1. TURBO LORA
# ============================================================

lora_target = (
    models_root
    / "loras"
    / TURBO_LORA_FILE
)

lora_target.parent.mkdir(
    parents=True,
    exist_ok=True,
)


if (
    not lora_target.exists()
    or file_gib(lora_target) < 0.70
):

    print(
        f"\nDownloading Turbo LoRA:\n"
        f"{TURBO_LORA_FILE}"
    )

    downloaded_lora = Path(
        hf_hub_download(
            repo_id=TURBO_REPO,
            filename=TURBO_LORA_FILE,
            local_dir=models_root / "loras",
            token=(
                HF_TOKEN
                or os.environ.get("HF_TOKEN")
                or None
            ),
        )
    )

    if (
        not downloaded_lora.exists()
        or file_gib(downloaded_lora) < 0.70
    ):
        raise RuntimeError(
            "Turbo LoRA download verification failed."
        )

    print(
        f"✓ LoRA: {file_gib(downloaded_lora):.2f} GiB"
    )

else:
    print(
        f"✓ LoRA already present: "
        f"{file_gib(lora_target):.2f} GiB"
    )


# ============================================================
# 2. H3 MODEL FILES
# ============================================================

for relative_path, minimum_gib in MODEL_FILES.items():

    target = models_root / relative_path

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Already valid
    if (
        target.exists()
        and file_gib(target) >= minimum_gib
    ):
        print(
            f"✓ {relative_path}: "
            f"{file_gib(target):.2f} GiB"
        )
        continue

    # Remove incomplete download
    if target.exists():
        print(
            f"Removing incomplete file:\n"
            f"{target}"
        )
        target.unlink()

    print(
        f"\nDownloading:\n"
        f"{relative_path}"
    )

    downloaded = Path(
        hf_hub_download(
            repo_id=MODEL_REPO,
            filename=relative_path,
            local_dir=models_root,
            token=(
                HF_TOKEN
                or os.environ.get("HF_TOKEN")
                or None
            ),
        )
    )

    if (
        not downloaded.exists()
        or file_gib(downloaded) < minimum_gib
    ):
        raise RuntimeError(
            f"Download verification failed: "
            f"{relative_path}"
        )

    print(
        f"✓ {relative_path}: "
        f"{file_gib(downloaded):.2f} GiB"
    )


# ============================================================
# 3. MANIFEST
# ============================================================

manifest = {
    relative_path: {
        "path": str(
            models_root / relative_path
        ),
        "bytes": (
            models_root
            / relative_path
        ).stat().st_size,
    }
    for relative_path in MODEL_FILES
}

manifest_path = (
    FINAL_OUTPUT_DIR
    / "minimax_h3_model_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

total_gib = (
    sum(
        item["bytes"]
        for item in manifest.values()
    )
    / (1024 ** 3)
)

print("\n" + "=" * 70)
print("MODEL DOWNLOAD VERIFICATION")
print("=" * 70)

for relative_path in MODEL_FILES:
    target = models_root / relative_path
    print(
        f"{file_gib(target):6.2f} GiB  "
        f"{relative_path}"
    )

print(
    f"\nVerified H3 payload: "
    f"{total_gib:.2f} GiB"
)

print(
    f"Turbo LoRA: "
    f"{file_gib(lora_target):.2f} GiB"
)

print(
    f"\nManifest: {manifest_path}"
)


minimax_h3_turbo_v4_step600_ema.safetensors


minimax_h3_turbo_v4_step600_ema.safetens(…): reconstructing file:   0%|          |  0.00B /  780MB            

minimax_h3_turbo_v4_step600_ema.safetens(…): downloading bytes:           |  0.00B            

✓ LoRA: 0.73 GiB

Downloading:
diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors


diffusion_models/minimax_h3_fl2va_pruned(…): reconstructing file:   0%|          |  0.00B / 21.0GB            

...

vae/minimax_h3_video_vae_fp16.safetensor(…): downloading bytes:           |  0.00B            

✓ vae/minimax_h3_video_vae_fp16.safetensors: 4.85 GiB

Downloading:
vae/minimax_h3_audio_vae_fp32.safetensors


vae/minimax_h3_audio_vae_fp32.safetensor(…): reconstructing file:   0%|          |  0.00B /  605MB            

vae/minimax_h3_audio_vae_fp32.safetensor(…): downloading bytes:           |  0.00B            

✓ vae/minimax_h3_audio_vae_fp32.safetensors: 0.56 GiB

MODEL DOWNLOAD VERIFICATION
 19.53 GiB  diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors
 14.61 GiB  text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors
  4.85 GiB  vae/minimax_h3_video_vae_fp16.safetensors
  0.56 GiB  vae/minimax_h3_audio_vae_fp32.safetensors

Verified H3 payload: 39.55 GiB
Turbo LoRA: 0.73 GiB

Manifest: /kaggle/working/minimax_h3_model_manifest.json


## 6. Start ComfyUI in low-VRAM API mode

Refactored into reusable `start_comfy_server()` / `stop_comfy_server()` functions. The server is started once here; the segment-generation loop later in this notebook calls these same functions again to periodically recycle the process on very long, many-segment jobs (`RESTART_SERVER_EVERY_N_CHUNKS`).

In [9]:
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

import requests


SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"

PID_PATH = (
    SESSION_ROOT
    / "comfyui.pid"
)

SERVER_PROCESS = None


def tail_file(
    path: Path,
    line_count: int = 60,
) -> str:

    if not path.exists():
        return "(log file does not exist)"

    lines = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    return "\n".join(
        lines[-line_count:]
    )


def stop_comfy_server():
    """Stop any ComfyUI process this notebook started, on this port."""
    global SERVER_PROCESS

    if PID_PATH.exists():
        try:
            old_pid = int(PID_PATH.read_text().strip())
            os.kill(old_pid, signal.SIGTERM)
            time.sleep(3)
        except Exception:
            pass
        PID_PATH.unlink(missing_ok=True)

    if SERVER_PROCESS is not None:
        try:
            SERVER_PROCESS.terminate()
            SERVER_PROCESS.wait(timeout=10)
        except Exception:
            pass
        SERVER_PROCESS = None

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def start_comfy_server():
    """Launch ComfyUI and block until it responds, or raise with the log tail."""
    global SERVER_PROCESS

    os.chdir(COMFY_DIR)

    env = os.environ.copy()

    env["CUDA_VISIBLE_DEVICES"] = ",".join(
        str(i) for i in range(torch.cuda.device_count())
    )

    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    env["HF_HOME"] = str(SESSION_ROOT / "hf_home")

    command = [
        sys.executable,
        "main.py",
        "--listen",
        "127.0.0.1",
        "--port",
        str(SERVER_PORT),
        "--lowvram",
        "--preview-method",
        "none",
        "--disable-metadata",
        "--output-directory",
        str(COMFY_OUTPUT_DIR),
        "--temp-directory",
        str(COMFY_TEMP_DIR),
    ]

    print("+", " ".join(command))

    # Append mode: the log spans every (re)start so the whole multi-hour
    # session's history stays in one file, with a clear marker per start.
    log_handle = LOG_PATH.open("a", encoding="utf-8")
    log_handle.write(
        f"\n\n{'=' * 70}\n"
        f" (RE)START {time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"{'=' * 70}\n\n"
    )
    log_handle.flush()

    SERVER_PROCESS = subprocess.Popen(
        command,
        cwd=COMFY_DIR,
        env=env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )

    PID_PATH.write_text(
        str(SERVER_PROCESS.pid),
        encoding="utf-8",
    )

    deadline = time.time() + 300
    last_error = None

    while time.time() < deadline:

        if SERVER_PROCESS.poll() is not None:
            log_handle.flush()
            raise RuntimeError(
                "ComfyUI exited during startup.\n\n"
                + tail_file(LOG_PATH, 120)
            )

        try:
            response = requests.get(
                f"{SERVER_URL}/system_stats",
                timeout=3,
            )
            if response.ok:
                print(f"\n✓ ComfyUI is ready: {SERVER_URL}")
                return
            last_error = f"HTTP {response.status_code}"
        except Exception as exc:
            last_error = repr(exc)

        time.sleep(2)

    raise RuntimeError(
        "ComfyUI did not become ready.\n"
        f"Last error: {last_error}\n\n"
        + tail_file(LOG_PATH, 120)
    )


# ------------------------------------------------------------
# Launch the server for the first time (a previous run in this
# same session, if any, is stopped first).
# ------------------------------------------------------------

stop_comfy_server()
start_comfy_server()

+ /usr/bin/python3 main.py --listen 127.0.0.1 --port 8188 --lowvram --preview-method none --disable-metadata --output-directory /kaggle/working/minimax_h3_output --temp-directory /kaggle/temp/minimax_h3_session/temp

✓ ComfyUI is ready: http://127.0.0.1:8188


## 7. Verify that all required H3 nodes loaded

In [10]:
import requests
import json

object_info_response = requests.get(
    f"{SERVER_URL}/object_info",
    timeout=60,
)

object_info_response.raise_for_status()

OBJECT_INFO = object_info_response.json()


REQUIRED_NODE_TYPES = [
    "UNETLoader",
    "CLIPLoader",
    "VAELoader",
    "MiniMaxH3ImageToVideo",
    "LoadImage",
    "RandomNoise",
    "KSamplerSelect",
    "BasicScheduler",
    "BasicGuider",
    "SamplerCustomAdvanced",
    "VAEDecode",
    "VAEDecodeAudio",
    "CreateVideo",
    "SaveVideo",
]

missing = [
    node
    for node in REQUIRED_NODE_TYPES
    if node not in OBJECT_INFO
]

if missing:

    raise RuntimeError(
        "Required H3 nodes are missing:\n"
        + json.dumps(
            missing,
            indent=2,
        )
        + "\n\nComfyUI log:\n"
        + tail_file(
            LOG_PATH,
            150,
        )
    )


MULTIGPU_NODE_AVAILABLE = (
    "H3KaggleCLIPLoader"
    in OBJECT_INFO
)

TURBO_NODE_AVAILABLE = (
    "MiniMaxH3TurboLoRA"
    in OBJECT_INFO
    and "MiniMaxH3TurboSampler"
    in OBJECT_INFO
)


print("=" * 70)
print("COMFYUI NODE CHECK")
print("=" * 70)

print("✓ Official H3 nodes loaded (including LoadImage, used for segment continuation)")
print(
    "Experimental GPU-1 loader:",
    MULTIGPU_NODE_AVAILABLE,
)

print(
    "Turbo nodes:",
    TURBO_NODE_AVAILABLE,
)


if USE_TURBO_LORA and not TURBO_NODE_AVAILABLE:

    raise RuntimeError(
        "Turbo LoRA was requested, but the "
        "MiniMax-H3-Turbo custom nodes are unavailable.\n\n"
        + tail_file(
            LOG_PATH,
            150,
        )
    )


for node_name in [
    "MiniMaxH3ImageToVideo",
    "LoadImage",
    "CLIPLoader",
    "H3KaggleCLIPLoader",
    "MiniMaxH3TurboLoRA",
    "MiniMaxH3TurboSampler",
    "SaveVideo",
]:

    if node_name in OBJECT_INFO:

        print(
            f"\n{node_name} input schema:"
        )

        print(
            json.dumps(
                OBJECT_INFO[node_name].get(
                    "input",
                    {},
                ),
                indent=2,
                default=str,
            )[:7000]
        )

COMFYUI NODE CHECK
✓ Official H3 nodes loaded (including LoadImage, used for segment continuation)
Experimental GPU-1 loader: True
Turbo nodes: True

MiniMaxH3ImageToVideo input schema:
{
  "required": {
    "clip": [
      "CLIP",
      {}
    ],
    "vae": [
      "VAE",
      {}
    ],
    "prompt": [
      "STRING",
      {
        "multiline": true,
        "dynamicPrompts": true
      }
    ],
    "width": [
      "INT",
      {
        "default": 1344,
        "min": 32,
        "max": 16384,
        "step": 32
      }
    ],
    "height": [
      "INT",
      {
        "default": 768,
        "min": 32,
        "max": 16384,
        "step": 32
      }
    ],
    "length": [
      "INT",
      {
        "tooltip": "Frame count at 24 fps, snapped up to the model's 17k+5 grid (124 = ~5s; trained range is ~124-362, longer is untested)",
        "default": 124,
        "min": 5,
        "max": 3600,
        "step": 17
      }
    ]
  },
  "optional": {
    "first_frame": [
      "IM

## 8. Build the API workflow

`build_h3_workflow()` builds one segment's ComfyUI API graph. When `first_frame_image` is given — the previous segment's last decoded frame, already copied into ComfyUI's `input/` folder — a `LoadImage` node is wired into `MiniMaxH3ImageToVideo`'s optional `first_frame` input. This is the only continuity mechanism the official H3 node exposes today, and it's what this notebook uses to extend a video across segments.

In [11]:
import json
from pathlib import Path


# ============================================================
# MODEL NAMES
# ============================================================

DIFFUSION_NAME = "minimax_h3_fl2va_pruned_int8_convrot.safetensors"
TEXT_ENCODER_NAME = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
VIDEO_VAE_NAME = "minimax_h3_video_vae_fp16.safetensors"
AUDIO_VAE_NAME = "minimax_h3_audio_vae_fp32.safetensors"
TURBO_LORA_NAME = "minimax_h3_turbo_v4_step600_ema.safetensors"


def build_h3_workflow(
    *,
    prompt: str,
    width: int,
    height: int,
    duration_seconds: float,
    steps: int,
    seed: int,
    use_gpu1_clip: bool,
    filename_prefix: str,
    use_turbo_lora: bool = False,
    first_frame_image: str = None,
):
    """
    Build one segment's ComfyUI API graph.

    first_frame_image: filename of an image already sitting in
    ComfyUI's `input/` directory (e.g. the previous segment's last
    decoded frame). When given, MiniMaxH3ImageToVideo's optional
    `first_frame` input is wired to a LoadImage node reading that
    file, which is how this notebook extends a video across chained
    segments (the node has no other continuation mechanism).
    """

    if width % 32 != 0:
        raise ValueError("H3 width must be divisible by 32.")

    if height % 32 != 0:
        raise ValueError("H3 height must be divisible by 32.")

    if steps < 1:
        raise ValueError("Steps must be >= 1.")

    length = snap_h3_frames(duration_seconds)

    use_custom_clip = (
        use_gpu1_clip
        and torch.cuda.device_count() > 1
        and MULTIGPU_NODE_AVAILABLE
    )

    if use_custom_clip:
        clip_node = {
            "class_type": "H3KaggleCLIPLoader",
            "inputs": {
                "clip_name": TEXT_ENCODER_NAME,
                "device": "cuda:1",
            },
        }
    else:
        clip_node = {
            "class_type": "CLIPLoader",
            "inputs": {
                "clip_name": TEXT_ENCODER_NAME,
                "type": "minimax",
                "device": "default",
            },
        }

    model_source = ["6", 0]

    i2v_inputs = {
        "clip": ["13", 0],
        "vae": ["11", 0],
        "prompt": prompt,
        "width": int(width),
        "height": int(height),
        "length": int(length),
    }

    workflow = {
        "6": {
            "class_type": "UNETLoader",
            "inputs": {"unet_name": DIFFUSION_NAME, "weight_dtype": "default"},
        },
        "11": {
            "class_type": "VAELoader",
            "inputs": {"vae_name": VIDEO_VAE_NAME},
        },
        "13": clip_node,
        "15": {
            "class_type": "RandomNoise",
            "inputs": {"noise_seed": int(seed)},
        },
        "104": {
            "class_type": "MiniMaxH3ImageToVideo",
            "inputs": i2v_inputs,
        },
        "17": {
            "class_type": "KSamplerSelect",
            "inputs": {"sampler_name": "res_multistep"},
        },
        "16": {
            "class_type": "BasicGuider",
            "inputs": {"model": model_source, "conditioning": ["104", 0]},
        },
        "9": {
            "class_type": "BasicScheduler",
            "inputs": {
                "model": model_source,
                "scheduler": "simple",
                "steps": int(steps),
                "denoise": 1.0,
            },
        },
        "14": {
            "class_type": "SamplerCustomAdvanced",
            "inputs": {
                "noise": ["15", 0],
                "guider": ["16", 0],
                "sampler": ["17", 0],
                "sigmas": ["9", 0],
                "latent_image": ["104", 1],
            },
        },
        "10": {
            "class_type": "VAEDecode",
            "inputs": {"samples": ["14", 0], "vae": ["11", 0]},
        },
        "24": {
            "class_type": "VAELoader",
            "inputs": {"vae_name": AUDIO_VAE_NAME},
        },
        "23": {
            "class_type": "VAEDecodeAudio",
            "inputs": {"samples": ["14", 0], "vae": ["24", 0]},
        },
        "91": {
            "class_type": "CreateVideo",
            "inputs": {
                "images": ["10", 0],
                "audio": ["23", 0],
                "fps": 24.0,
                "bit_depth": 8,
            },
        },
        "92": {
            "class_type": "SaveVideo",
            "inputs": {
                "video": ["91", 0],
                "filename_prefix": filename_prefix,
                "format": "auto",
                "codec": "auto",
            },
        },
    }

    # ------------------------------------------------------------
    # Segment continuation: feed the previous segment's last frame
    # in as this segment's first_frame conditioning.
    # ------------------------------------------------------------
    if first_frame_image:
        workflow["105"] = {
            "class_type": "LoadImage",
            "inputs": {"image": first_frame_image},
        }
        workflow["104"]["inputs"]["first_frame"] = ["105", 0]

    # ------------------------------------------------------------
    # Turbo LoRA
    # ------------------------------------------------------------
    if use_turbo_lora:

        if not TURBO_NODE_AVAILABLE:
            raise RuntimeError(
                "Turbo LoRA was requested, but the "
                "MiniMax-H3-Turbo custom nodes are unavailable."
            )

        workflow["60"] = {
            "class_type": "MiniMaxH3TurboLoRA",
            "inputs": {
                "model": ["6", 0],
                "lora_name": TURBO_LORA_NAME,
                "strength": 1.0,
                "low_vram": True,
            },
        }

        model_source = ["60", 0]
        workflow["17"] = {"class_type": "MiniMaxH3TurboSampler", "inputs": {}}
        workflow["16"]["inputs"]["model"] = model_source
        workflow["9"]["inputs"]["model"] = model_source

    config = {
        "width": width,
        "height": height,
        "duration_requested": duration_seconds,
        "frames": length,
        "actual_duration": length / 24,
        "steps": steps,
        "seed": seed,
        "dual_gpu_clip": use_custom_clip,
        "use_turbo_lora": use_turbo_lora,
        "continuation_frame": first_frame_image,
    }

    return workflow, config


# ============================================================
# PREVIEW / VALIDATE WORKFLOW (segment 1, no continuation yet)
# ============================================================

preview_workflow, preview_config = build_h3_workflow(
    prompt=PROMPT,
    width=WIDTH,
    height=HEIGHT,
    duration_seconds=SEGMENTS[0]["requested_seconds"],
    steps=STEPS,
    seed=SEED,
    use_gpu1_clip=USE_SECOND_T4_FOR_TEXT_ENCODER,
    filename_prefix=SAVE_PREFIX,
    use_turbo_lora=USE_TURBO_LORA,
)

workflow_path = FINAL_OUTPUT_DIR / "minimax_h3_api_workflow.json"

workflow_path.write_text(
    json.dumps(preview_workflow, indent=2),
    encoding="utf-8",
)

print(json.dumps(preview_config, indent=2))
print("\nSaved workflow:", workflow_path)

if len(SEGMENTS) > 1:
    print(
        f"\nNote: this preview shows segment 1 of {len(SEGMENTS)}. "
        f"Later segments add a 'first_frame' continuation input automatically."
    )

{
  "width": 352,
  "height": 608,
  "duration_requested": 10.0,
  "frames": 243,
  "actual_duration": 10.125,
  "steps": 4,
  "seed": 20260803,
  "dual_gpu_clip": false,
  "use_turbo_lora": true,
  "continuation_frame": null
}

Saved workflow: /kaggle/working/minimax_h3_api_workflow.json

Note: this preview shows segment 1 of 3. Later segments add a 'first_frame' continuation input automatically.


## 9. Generate

For each planned segment (see the profile card above), this cell:

- builds the H3 workflow, continuing from the previous segment's last frame automatically when this isn't the first segment;
- shows the current workflow stage and a live H3 sampling progress bar with step count and ETA;
- retries a segment with progressively safer settings (lower resolution/steps, and if needed a shorter chunk) **only** on genuine CUDA out-of-memory errors — this is what keeps a long, many-segment job reliable ("took as long as hours, but never died to OOM") instead of failing partway through;
- **skips segments that already completed in a previous run** of this cell, tracked in `minimax_h3_segments/segment_state.json` — so a Kaggle disconnect/reconnect resumes instead of starting the whole video over;
- optionally recycles the whole ComfyUI process every `RESTART_SERVER_EVERY_N_CHUNKS` segments, to reset GPU/RAM state on very long jobs.

A request at or under `MAX_SINGLE_SHOT_SECONDS` runs as exactly one segment here, with no hand-off logic at all — identical in behavior/quality to a plain single-shot generation.

In [12]:
import json
import shutil
import subprocess
import time
import uuid
from pathlib import Path

import requests
import websocket

from IPython.display import HTML, display
from tqdm.auto import tqdm


VIDEO_EXTENSIONS = {
    ".mp4",
    ".webm",
    ".mkv",
    ".mov",
}


NODE_STAGES = {
    "6": "Loading MiniMax H3 diffusion model",
    "11": "Loading video VAE",
    "13": "Encoding prompt",
    "15": "Preparing random noise",
    "17": "Preparing sampler",
    "104": "Building H3 audio-video latent",
    "105": "Loading continuation frame from previous segment",
    "16": "Preparing conditioning",
    "9": "Preparing sigma schedule",
    "14": "Sampling H3 video + stereo audio",
    "24": "Loading audio VAE",
    "23": "Decoding audio",
    "10": "Decoding video",
    "91": "Combining video + audio",
    "92": "Saving final video",
}


def collect_videos():
    return {
        p.resolve()
        for p in COMFY_OUTPUT_DIR.rglob("*")
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    }


def free_comfy_memory():
    try:
        requests.post(
            f"{SERVER_URL}/free",
            json={"unload_models": True, "free_memory": True},
            timeout=30,
        )
    except Exception:
        pass

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def queue_workflow(workflow, client_id):
    response = requests.post(
        f"{SERVER_URL}/prompt",
        json={"prompt": workflow, "client_id": client_id},
        timeout=120,
    )

    if not response.ok:
        raise RuntimeError(
            "ComfyUI rejected workflow:\n\n"
            + response.text
            + "\n\nLog:\n"
            + tail_file(LOG_PATH, 120)
        )

    data = response.json()

    if "prompt_id" not in data:
        raise RuntimeError(f"Unexpected ComfyUI response:\n{data}")

    return data["prompt_id"]


def read_prompt_history(prompt_id):
    response = requests.get(
        f"{SERVER_URL}/history/{prompt_id}",
        timeout=30,
    )
    response.raise_for_status()
    return response.json().get(prompt_id)


def status_card(title, detail="", state="running"):
    palette = {
        "running": ("#70d85c", "rgba(70,150,60,.10)"),
        "success": ("#54d97b", "rgba(50,160,90,.12)"),
        "error": ("#ef6a6a", "rgba(180,50,50,.10)"),
        "waiting": ("#d5b85c", "rgba(160,130,40,.10)"),
    }

    accent, background = palette[state]

    return HTML(
        f"""
        <div style="
            border:1px solid {accent};
            border-left:7px solid {accent};
            border-radius:8px;
            padding:12px 15px;
            margin:8px 0 10px 0;
            background:{background};
            font-family:system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
        ">
          <div style="font-size:16px;font-weight:750;">{title}</div>
          <div style="margin-top:4px;opacity:.76;">{detail}</div>
        </div>
        """
    )


def wait_for_prompt(prompt_id, expected_steps, ws=None):

    started = time.time()

    status_handle = display(
        status_card(
            "Waiting for ComfyUI",
            "Initial model loading can take several minutes.",
            "waiting",
        ),
        display_id=True,
    )

    progress_bar = tqdm(
        total=int(expected_steps),
        desc="H3 sampling",
        unit="step",
        dynamic_ncols=True,
        leave=True,
        mininterval=0.2,
    )

    def update_stage(stage, state="running"):
        elapsed = time.time() - started
        status_handle.update(
            status_card(stage, f"Elapsed: {elapsed / 60:.1f} min", state)
        )
        fraction = progress_bar.n / max(progress_bar.total, 1)
        report_automation_progress(
            stage, f"Elapsed: {elapsed / 60:.1f} min",
            AUTOMATION_ACTIVE_SEGMENT_INDEX,
            AUTOMATION_ACTIVE_SEGMENT_INDEX,
            len(SEGMENTS), fraction,
        )

    try:

        while True:

            message = None

            if ws is not None:
                try:
                    raw = ws.recv()
                    if isinstance(raw, str):
                        message = json.loads(raw)
                except websocket.WebSocketTimeoutException:
                    message = None
                except Exception:
                    ws = None
                    update_stage(
                        "Live progress disconnected",
                        "Falling back to HTTP history.",
                        "waiting",
                    )

            if message is not None:

                message_type = message.get("type")
                data = message.get("data", {})
                message_prompt_id = data.get("prompt_id")

                if message_prompt_id is not None and message_prompt_id != prompt_id:
                    continue

                if message_type == "executing":
                    node = data.get("node")
                    if node is None:
                        break
                    update_stage(NODE_STAGES.get(str(node), "Running H3 workflow"))

                elif message_type == "progress":
                    node = str(data.get("node", ""))
                    if node == "14":
                        maximum = int(data.get("max", expected_steps))
                        value = int(data.get("value", 0))
                        if maximum > 0 and progress_bar.total != maximum:
                            progress_bar.total = maximum
                        value = min(value, progress_bar.total)
                        if value > progress_bar.n:
                            progress_bar.update(value - progress_bar.n)
                        update_stage("Sampling H3 video + stereo audio")

                elif message_type in ("execution_error", "execution_interrupted"):
                    raise RuntimeError(
                        "ComfyUI execution failed:\n"
                        + json.dumps(data, indent=2, default=str)[-12000:]
                        + "\n\nLog:\n"
                        + tail_file(LOG_PATH, 120)
                    )

                elif message_type == "execution_success":
                    break

            history = read_prompt_history(prompt_id)

            if history is not None:
                status = history.get("status", {})

                if status.get("status_str") == "error":
                    raise RuntimeError(
                        "ComfyUI execution failed:\n"
                        + json.dumps(status.get("messages", []), indent=2, default=str)[-12000:]
                        + "\n\nLog:\n"
                        + tail_file(LOG_PATH, 120)
                    )

                if status.get("completed", False):
                    if progress_bar.n < progress_bar.total:
                        progress_bar.update(progress_bar.total - progress_bar.n)
                    update_stage("Generation completed", "success")
                    return history

            if ws is None:
                time.sleep(3)

        for _ in range(20):

            history = read_prompt_history(prompt_id)

            if history is not None:
                status = history.get("status", {})

                if status.get("status_str") == "error":
                    raise RuntimeError(
                        "ComfyUI execution failed:\n"
                        + json.dumps(status.get("messages", []), indent=2, default=str)[-12000:]
                        + "\n\nLog:\n"
                        + tail_file(LOG_PATH, 120)
                    )

                if status.get("completed", False):
                    if progress_bar.n < progress_bar.total:
                        progress_bar.update(progress_bar.total - progress_bar.n)
                    update_stage("Generation completed", "success")
                    return history

            time.sleep(0.5)

        raise RuntimeError(
            "ComfyUI announced completion, but no final history record appeared."
        )

    except Exception:
        status_handle.update(
            status_card(
                "Generation failed",
                "See the diagnostics cell for the ComfyUI traceback.",
                "error",
            )
        )
        raise

    finally:
        progress_bar.close()


# ============================================================
# Segment execution, with per-segment OOM fallback and resumability
# ============================================================

AUTOMATION_ACTIVE_SEGMENT_INDEX = 0
SEGMENTS_DIR = FINAL_OUTPUT_DIR / "minimax_h3_segments"
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = SEGMENTS_DIR / "segment_state.json"


def load_segment_state():
    if STATE_PATH.is_file():
        try:
            return json.loads(STATE_PATH.read_text(encoding="utf-8"))
        except Exception:
            return {}
    return {}


def save_segment_state(state):
    STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")


SEGMENT_STATE = load_segment_state()


def get_segment_prompt(index):
    overrides = SEGMENT_PROMPTS
    if overrides:
        return str(overrides[index] if index < len(overrides) else overrides[-1]).strip()
    return PROMPT


def is_memory_failure(error_text: str) -> bool:
    text = error_text.lower()
    markers = (
        "out of memory",
        "cuda out of memory",
        "cuda error: memory",
        "allocation failed",
        "cublas_status_alloc_failed",
        "not enough memory",
    )
    return any(marker in text for marker in markers)


def build_attempt_tiers(segment):
    base = {
        "width": WIDTH,
        "height": HEIGHT,
        "duration": segment["requested_seconds"],
        "steps": STEPS,
        "gpu1_clip": USE_SECOND_T4_FOR_TEXT_ENCODER,
        "use_turbo": USE_TURBO_LORA,
        "seed": SEED + segment["index"],
    }

    tiers = [dict(base, label=f"seg{segment['index']:02d}_primary")]
    fallback_width, fallback_height = ((288, 512) if HEIGHT > WIDTH else (512, 288))

    if AUTO_FALLBACK:
        tiers.append(dict(
            base,
            label=f"seg{segment['index']:02d}_lowvram_fallback",
            width=fallback_width,
            height=fallback_height,
            steps=(8 if USE_TURBO_LORA else 12),
            gpu1_clip=False,
        ))
        tiers.append(dict(
            base,
            label=f"seg{segment['index']:02d}_min_fallback",
            width=fallback_width,
            height=fallback_height,
            steps=(6 if USE_TURBO_LORA else 8),
            gpu1_clip=False,
        ))

    return tiers


def execute_attempt(attempt, first_frame_image, prompt_text):

    safe_label = attempt["label"].replace(" ", "_").replace("/", "_")
    prefix = f"{SAVE_PREFIX}_{safe_label}"

    workflow, resolved = build_h3_workflow(
        prompt=prompt_text,
        width=attempt["width"],
        height=attempt["height"],
        duration_seconds=attempt["duration"],
        steps=attempt["steps"],
        seed=attempt["seed"],
        use_gpu1_clip=attempt["gpu1_clip"],
        filename_prefix=prefix,
        use_turbo_lora=attempt["use_turbo"],
        first_frame_image=first_frame_image,
    )

    display(status_card(
        "Attempt",
        f"{resolved['width']}×{resolved['height']} · {resolved['actual_duration']:.2f}s · "
        f"{resolved['steps']} steps · Turbo={'yes' if resolved['use_turbo_lora'] else 'no'}"
        + (" · continuation frame" if first_frame_image else ""),
        "running",
    ))

    before = collect_videos()
    client_id = str(uuid.uuid4())
    ws = None

    try:
        ws = websocket.WebSocket()
        ws.settimeout(5)
        ws.connect(f"ws://127.0.0.1:{SERVER_PORT}/ws?clientId={client_id}", timeout=30)
    except Exception:
        ws = None

    try:
        prompt_id = queue_workflow(workflow, client_id)
        history = wait_for_prompt(prompt_id, expected_steps=attempt["steps"], ws=ws)
    finally:
        if ws is not None:
            try:
                ws.close()
            except Exception:
                pass

    after = collect_videos()
    new_videos = sorted(after - before, key=lambda p: p.stat().st_mtime)

    if not new_videos:
        new_videos = sorted(after, key=lambda p: p.stat().st_mtime)

    if not new_videos:
        raise RuntimeError(
            "ComfyUI completed, but no video was found.\n\n" + tail_file(LOG_PATH, 120)
        )

    return new_videos[-1], resolved, prompt_id, history


def extract_last_frame(video_path, frame_count, dst_png):
    cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", str(video_path),
        "-vf", rf"select=eq(n\,{max(frame_count - 1, 0)})",
        "-vsync", "0", "-frames:v", "1",
        str(dst_png),
    ]
    subprocess.run([str(c) for c in cmd], check=True)

    if not dst_png.is_file():
        raise RuntimeError(f"Could not extract last frame from {video_path}")


def run_segment(segment):

    global AUTOMATION_ACTIVE_SEGMENT_INDEX
    index = segment["index"]
    AUTOMATION_ACTIVE_SEGMENT_INDEX = index
    key = str(index)
    report_automation_progress(
        "Preparing clip", f"Preparing clip {index + 1} of {len(SEGMENTS)}",
        index, index, len(SEGMENTS), 0.0,
    )

    existing = SEGMENT_STATE.get(key)
    if existing and Path(existing["video_path"]).is_file():
        print(f"✓ Segment {index + 1}/{len(SEGMENTS)} already completed — reusing {existing['video_path']}")
        return existing

    is_continuation = not segment["is_first"]

    display(status_card(
        f"Segment {index + 1}/{len(SEGMENTS)}",
        f"Target {segment['actual_seconds']:.2f}s ({segment['frames']} frames)"
        + (" · continuing from previous segment's last frame" if is_continuation else ""),
        "running",
    ))

    continuation_image = None
    if is_continuation:
        prev = SEGMENT_STATE[str(index - 1)]
        continuation_image = prev["last_frame_comfy_name"]

    tiers = build_attempt_tiers(segment)
    prompt_text = get_segment_prompt(index)
    errors = []
    success = None

    for tier_index, attempt in enumerate(tiers):
        try:
            success = execute_attempt(attempt, continuation_image, prompt_text)
            break
        except Exception as exc:
            error_text = f"{type(exc).__name__}: {exc}"
            errors.append({"attempt": attempt, "error": error_text})

            has_more = tier_index + 1 < len(tiers)
            if not (AUTO_FALLBACK and has_more and is_memory_failure(error_text)):
                break

            display(status_card(
                "VRAM limit reached",
                "Unloading models and retrying this segment with safer settings.",
                "waiting",
            ))
            free_comfy_memory()
            time.sleep(5)

    if success is None:
        error_path = SEGMENTS_DIR / f"segment_{index:02d}_errors.json"
        error_path.write_text(json.dumps(errors, indent=2), encoding="utf-8")
        raise RuntimeError(
            f"Segment {index + 1}/{len(SEGMENTS)} failed after {len(tiers)} attempt(s).\n\n"
            + (errors[-1]["error"] if errors else "No error details were captured.")
        )

    generated_path, resolved, prompt_id, history = success

    segment_video = SEGMENTS_DIR / f"segment_{index:02d}.mp4"
    shutil.copy2(generated_path, segment_video)

    last_frame_png = SEGMENTS_DIR / f"segment_{index:02d}_last_frame.png"
    extract_last_frame(segment_video, resolved["frames"], last_frame_png)

    comfy_input_dir = COMFY_DIR / "input"
    comfy_input_dir.mkdir(parents=True, exist_ok=True)
    comfy_frame_name = f"h3_continuation_seg{index:02d}.png"
    shutil.copy2(last_frame_png, comfy_input_dir / comfy_frame_name)

    record = {
        "index": index,
        "video_path": str(segment_video),
        "last_frame_png": str(last_frame_png),
        "last_frame_comfy_name": comfy_frame_name,
        "resolved_config": resolved,
        "prompt_id": prompt_id,
    }
    SEGMENT_STATE[key] = record
    save_segment_state(SEGMENT_STATE)
    report_automation_progress(
        "Clip complete", f"Clip {index + 1} of {len(SEGMENTS)} is complete",
        index, index + 1, len(SEGMENTS), 0.0,
    )

    display(status_card(
        f"Segment {index + 1}/{len(SEGMENTS)} complete",
        f"Saved {segment_video.name}",
        "success",
    ))

    return record


overall_started = time.time()
print(f"Generating {len(SEGMENTS)} segment(s), ~{TOTAL_ACTUAL_SECONDS:.2f}s total...")

for seg in SEGMENTS:

    run_segment(seg)

    is_last = seg["index"] + 1 == len(SEGMENTS)

    if RESTART_SERVER_EVERY_N_CHUNKS and not is_last and (seg["index"] + 1) % RESTART_SERVER_EVERY_N_CHUNKS == 0:
        display(status_card(
            "Recycling ComfyUI server",
            "Restarting to reset GPU/RAM state before the next segment.",
            "waiting",
        ))
        stop_comfy_server()
        start_comfy_server()
    else:
        free_comfy_memory()

print(f"\n✓ All {len(SEGMENTS)} segment(s) complete in {(time.time() - overall_started) / 60:.1f} min.")

Generating 3 segment(s), ~30.38s total...


H3 sampling:   0%|          | 0/4 [00:00<?, ?step/s]

H3 sampling:   0%|          | 0/4 [00:00<?, ?step/s]

H3 sampling:   0%|          | 0/4 [00:00<?, ?step/s]


✓ All 3 segment(s) complete in 64.8 min.


## 10. Stitch segments into the final video

A single-segment run (request ≤ `MAX_SINGLE_SHOT_SECONDS`) is copied straight through with **zero re-encoding** — full original quality, identical to a plain single-shot generation.

Multiple segments are joined with FFmpeg, trimming the one duplicated hand-off frame at each seam — in *both* the video and the matching sliver of audio, so sync is preserved — and re-encoding once using the CRF/preset/pixel-format settings from the advanced settings cell.

In [13]:
import json
import shutil
import subprocess
from pathlib import Path

missing = [i for i in range(len(SEGMENTS)) if str(i) not in SEGMENT_STATE]
if missing:
    raise RuntimeError(
        f"Segment(s) {missing} have not completed yet — run the generation cell first."
    )

ordered_segments = [SEGMENT_STATE[str(i)] for i in range(len(SEGMENTS))]
report_automation_progress(
    "Stitching final video", "Joining generated clips into the final video",
    len(SEGMENTS) - 1, len(SEGMENTS), len(SEGMENTS), 0.0,
)
FPS = 24.0

if len(ordered_segments) == 1:

    only = ordered_segments[0]
    final_suffix = Path(only["video_path"]).suffix.lower() or ".mp4"
    FINAL_VIDEO = FINAL_OUTPUT_DIR / f"minimax_h3_final{final_suffix}"
    shutil.copy2(only["video_path"], FINAL_VIDEO)
    print(f"✓ Single segment — copied directly with no re-encode: {FINAL_VIDEO}")

else:

    trim_offset = 1.0 / FPS
    ffmpeg_inputs, filter_parts = [], []

    for i, seg in enumerate(ordered_segments):

        ffmpeg_inputs += ["-i", str(seg["video_path"])]

        if i == 0:
            filter_parts.append(f"[{i}:v]setpts=PTS-STARTPTS[v{i}];")
            filter_parts.append(f"[{i}:a]asetpts=PTS-STARTPTS[a{i}];")
        else:
            # Drop the single duplicated frame at the start of every
            # continuation segment (it is the same image that conditioned
            # this segment's first frame), so the hand-off doesn't stutter.
            filter_parts.append(f"[{i}:v]trim=start={trim_offset:.6f},setpts=PTS-STARTPTS[v{i}];")
            filter_parts.append(f"[{i}:a]atrim=start={trim_offset:.6f},asetpts=PTS-STARTPTS[a{i}];")

    concat_labels = "".join(f"[v{i}][a{i}]" for i in range(len(ordered_segments)))
    filter_complex = (
        "".join(filter_parts)
        + f"{concat_labels}concat=n={len(ordered_segments)}:v=1:a=1[vout][aout]"
    )

    FINAL_VIDEO = FINAL_OUTPUT_DIR / "minimax_h3_final.mp4"

    ffmpeg_command = (
        ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y"]
        + ffmpeg_inputs
        + [
            "-filter_complex", filter_complex,
            "-map", "[vout]", "-map", "[aout]",
            "-c:v", "libx264",
            "-crf", str(VIDEO_CRF),
            "-preset", str(VIDEO_PRESET),
            "-pix_fmt", str(VIDEO_PIXEL_FORMAT),
        ]
    )

    if VIDEO_FASTSTART:
        ffmpeg_command += ["-movflags", "+faststart"]

    ffmpeg_command += ["-c:a", "aac", "-b:a", "192k", str(FINAL_VIDEO)]

    print("+", " ".join(map(str, ffmpeg_command)))
    result = subprocess.run(ffmpeg_command, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError("FFmpeg segment stitching failed.")

    print(f"✓ Stitched {len(ordered_segments)} segments into: {FINAL_VIDEO}")

if not FINAL_VIDEO.is_file() or FINAL_VIDEO.stat().st_size == 0:
    raise RuntimeError("Final video was not produced correctly.")

print(f"  Size: {FINAL_VIDEO.stat().st_size / (1024 ** 2):.2f} MiB")

run_metadata = {
    "prompt": PROMPT,
    "segment_prompts": SEGMENT_PROMPTS,
    "requested_duration_seconds": DURATION_SECONDS,
    "planned_actual_duration_seconds": TOTAL_ACTUAL_SECONDS,
    "segments": [
        {
            "index": s["index"],
            "video_path": s["video_path"],
            "resolved_config": s["resolved_config"],
        }
        for s in ordered_segments
    ],
    "num_segments": len(ordered_segments),
    "comfyui_commit": COMFY_COMMIT,
    "final_video": str(FINAL_VIDEO),
    "model_repo": MODEL_REPO,
    "model_files": list(MODEL_FILES),
    "turbo_lora": TURBO_LORA_NAME if USE_TURBO_LORA else None,
}

(FINAL_OUTPUT_DIR / "minimax_h3_run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

display(status_card("Final video is ready", f"Saved to {FINAL_VIDEO}", "success"))

+ ffmpeg -hide_banner -loglevel error -y -i /kaggle/working/minimax_h3_segments/segment_00.mp4 -i /kaggle/working/minimax_h3_segments/segment_01.mp4 -i /kaggle/working/minimax_h3_segments/segment_02.mp4 -filter_complex [0:v]setpts=PTS-STARTPTS[v0];[0:a]asetpts=PTS-STARTPTS[a0];[1:v]trim=start=0.041667,setpts=PTS-STARTPTS[v1];[1:a]atrim=start=0.041667,asetpts=PTS-STARTPTS[a1];[2:v]trim=start=0.041667,setpts=PTS-STARTPTS[v2];[2:a]atrim=start=0.041667,asetpts=PTS-STARTPTS[a2];[v0][a0][v1][a1][v2][a2]concat=n=3:v=1:a=1[vout][aout] -map [vout] -map [aout] -c:v libx264 -crf 18 -preset slow -pix_fmt yuv420p -movflags +faststart -c:a aac -b:a 192k /kaggle/working/minimax_h3_final.mp4
✓ Stitched 3 segments into: /kaggle/working/minimax_h3_final.mp4
  Size: 2.48 MiB


## 11. Preview and keep the result

In [14]:
from IPython.display import Video, display
from pathlib import Path

if not Path(FINAL_VIDEO).exists():
    raise FileNotFoundError(
        f"Final video not found: {FINAL_VIDEO}"
    )

size_mib = (
    Path(FINAL_VIDEO).stat().st_size
    / (1024 ** 2)
)

print(
    f"Output size: {size_mib:.2f} MiB"
)

display(
    Video(
        str(FINAL_VIDEO),
        embed=True,
    )
)

Output size: 2.48 MiB


## 12. Optional diagnostics

Run this cell only when troubleshooting. The main generation cell intentionally keeps raw GPU telemetry and ComfyUI logs out of view.

In [15]:
print("=" * 70)
print("GPU STATUS")
print("=" * 70)

subprocess.run(
    ["nvidia-smi"],
    check=False,
)

print("\n" + "=" * 70)
print("LAST 150 COMFYUI LOG LINES")
print("=" * 70)

print(
    tail_file(
        LOG_PATH,
        150,
    )
)

print("\n" + "=" * 70)
print("SEGMENTS")
print("=" * 70)

for path in sorted(SEGMENTS_DIR.glob("segment_*.mp4")):
    print(f"- {path.name}: {path.stat().st_size / (1024 ** 2):.2f} MiB")

print("\n" + "=" * 70)
print("MINIMAX H3 OUTPUT FILES")
print("=" * 70)

for path in sorted(
    FINAL_OUTPUT_DIR.glob(
        "minimax_h3*"
    )
):

    if path.is_file():

        print(
            f"- {path.name}: "
            f"{path.stat().st_size / (1024 ** 2):.2f} MiB"
        )

GPU STATUS
Thu Sep 24 14:06:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             30W /   70W |     267MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------------

## Automation manifest

Writes a single JSON summary of this run's outputs — the one file an external automation needs to read first.

In [16]:
# ============================================================
# AUTOMATION MANIFEST
# ============================================================
#
# Single JSON file describing exactly what this run produced, so an
# external automation only needs to read one file to know what to
# download and what prompt/settings/segment-plan produced it.

import json
import time
from pathlib import Path


def _file_info(path):
    path = Path(path)
    if not path.is_file():
        return None
    return {"path": str(path), "size_bytes": path.stat().st_size}


manifest = {
    "job_id": PARAMS.get("job_id"),
    "prompt": PROMPT,
    "segment_prompts": SEGMENT_PROMPTS,
    "requested_duration_seconds": DURATION_SECONDS,
    "planned_actual_duration_seconds": TOTAL_ACTUAL_SECONDS,
    "num_segments": len(SEGMENTS),
    "max_single_shot_seconds": MAX_SINGLE_SHOT_SECONDS,
    "chunk_seconds": CHUNK_SECONDS,
    "seed": SEED,
    "segments": [
        {
            "index": s["index"],
            "video": _file_info(s["video_path"]),
            "resolved_config": s["resolved_config"],
            "prompt_id": s["prompt_id"],
        }
        for s in [SEGMENT_STATE[str(i)] for i in range(len(SEGMENTS))]
    ],
    "final_video": _file_info(FINAL_VIDEO),
    "workflow_json": _file_info(FINAL_OUTPUT_DIR / "minimax_h3_api_workflow.json"),
    "run_metadata_json": _file_info(FINAL_OUTPUT_DIR / "minimax_h3_run_metadata.json"),
    "comfyui_log": _file_info(LOG_PATH),
    "comfyui_commit": COMFY_COMMIT,
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "status": "success",
}

manifest_path = Path("/kaggle/working/automation_manifest.json")
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
report_automation_progress(
    "Notebook complete", "Kaggle finished the final video",
    len(SEGMENTS) - 1, len(SEGMENTS), len(SEGMENTS), 0.0,
)

print(f"✓ Automation manifest written: {manifest_path}")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

✓ Automation manifest written: /kaggle/working/automation_manifest.json
{
  "job_id": "h3-30s-night-lamp",
  "prompt": "integrated_multimodal_description:\n\n[Segment 1, 0–10s]\n\nVertical 9:16 ultra-realistic live-action cinematic science-fiction thriller.\n\nA woman in her late twenties stands alone beneath a covered bus stop on a quiet residential street at night. She has a realistic natural face, dark shoulder-length hair, realistic skin texture, realistic eyes, a dark wool coat, and a simple smartwatch on her left wrist.\n\nCHARACTER CONTINUITY:\nKeep exactly the same woman throughout the entire sequence. Preserve the same face, facial identity, hairstyle, hair length, dark wool coat, smartwatch, body proportions, skin texture, and overall silhouette. No character transformation, no clothing change, no hairstyle change, no age change, no duplicated person.\n\nThe street is quiet and mostly empty. Light mist drifts naturally through the environment. A single overhead streetlamp abo

## Optional cleanup — disabled by default

Only clean up `minimax_h3_segments/` after you have verified the final video and no longer need resume capability.

In [17]:
# Uncomment only after you have confirmed the final video is good.
#
# import shutil
#
# shutil.rmtree(SEGMENTS_DIR, ignore_errors=True)
# print("Deleted minimax_h3_segments/ (per-segment videos, last-frame PNGs, and state) to reclaim disk space.")

## Completed pipeline

Your final file is the MiniMax H3 video, automatically extended (if requested) via chained segment generation.

- `/kaggle/working/minimax_h3_final.mp4` → **final video**, at your requested duration
- `/kaggle/working/minimax_h3_segments/` → per-segment source videos, last-frame hand-off PNGs, and `segment_state.json` (kept for resumability; safe to delete once you've confirmed the final video, via the optional cleanup cell above)
- `/kaggle/working/automation_manifest.json` → **one-file summary for external automation** (prompt, resolved settings, seed, segment plan, and every output file's path + size)
- `/kaggle/working/minimax_h3_run_metadata.json` → detailed per-segment resolved configuration (resolution/steps/seed/turbo actually used for each segment, useful when a fallback tier kicked in)
- `/kaggle/working/minimax_h3_comfyui.log` → full ComfyUI server log across every (re)start, for troubleshooting

### Quick reference: what changes for different requested durations

| `duration_seconds` | Segments | Chaining | Notes |
|---|---|---|---|
| 10 | 1 | none | identical behavior to a plain single-shot run |
| 15 | 1 | none | still fits the model's trained-safe single-shot range |
| 20 | 2 | 1 hand-off | ~10s each, automatically |
| 60 | 4 | 3 hand-offs | ~15s each, automatically |
| any other value | `ceil(duration / max_single_shot_seconds)` | auto | override with `chunk_seconds` for an exact per-segment length instead |